# CS4120 Final — Part 2 Preprocessing: FIREBALL → our JSON schema

Run this on **free Colab**. It downloads FIREBALL's `filtered_triples.jsonl`, parses it with our schema, and produces `train.jsonl / dev.jsonl / test.jsonl` in `/content/processed/`.

**Expected time on free Colab:** ~5-10 minutes for download + ~1-2 minutes for parsing.

**Expected disk use:** FIREBALL's filtered_triples is a few hundred MB; processed output is ~50-100 MB depending on filters.

---

## What this notebook does
1. Install `huggingface_hub` and `datasets`
2. Download `filtered_triples.jsonl` from `lara-martin/FIREBALL`
3. Run `fireball_preprocess.run_pipeline` to convert to our schema
4. Print dataset statistics and a few sample records
5. Save outputs to `/content/processed/` (and optionally to your Google Drive)

## 1. Install dependencies

In [1]:
# deps installed in local venv


## 2. Upload `fireball_preprocess.py`

In the Colab sidebar, click the **Files** icon and drag `fireball_preprocess.py` into `/content/`. Or run the cell below to pull it from a public URL if you put it on GitHub.

In [2]:
import os
assert os.path.exists('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/fireball_preprocess.py'), 'fireball_preprocess.py missing'
print('OK')


OK


## 3. Download FIREBALL `filtered_triples.jsonl`

In [3]:
import os
path = '/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/raw/fireball_150combats_with_ids.jsonl'
assert os.path.exists(path), f'raw file missing: {path}'
print('Using local raw file:', path)


Using local raw file: /Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/raw/fireball_150combats_with_ids.jsonl


In [4]:
# Sanity check: size and the first line
import os, json
print(f'File size: {os.path.getsize(path) / 1e6:.1f} MB')
with open(path) as f:
    first = json.loads(f.readline())
print('Fields in a triple:', sorted(first.keys()))
print('\nExample after_utterances:', first.get('after_utterances'))
print('\nExample commands_norm:', first.get('commands_norm'))
print('\nExample automation_results[:2]:', (first.get('automation_results') or [])[:2])

File size: 210.9 MB
Fields in a triple: ['after_idxs', 'after_state_idx', 'after_utterances', 'automation_results', 'before_idxs', 'before_state_idx', 'before_utterances', 'caster_after', 'combat_id', 'combat_state_after', 'combat_state_before', 'command_idxs', 'commands_norm', 'current_actor', 'embed_idxs', 'speaker_id', 'targets_after', 'utterance_history']

Example after_utterances: []

Example commands_norm: ['!cast armor -t nim']

Example automation_results[:2]: ['Nimue Willowleaf casts Mage Armor!\nNimue Willowleaf gained Mage Armor.']


## 4. Run the preprocessing pipeline

In [5]:
import sys; sys.path.insert(0, '/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2')
from fireball_preprocess import run_pipeline

stats = run_pipeline(
    input_path=path,
    out_dir='/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/processed',
    max_records=0,          # 0 = no cap. Set to e.g. 20000 for a quick pilot
    train_frac=0.90,
    dev_frac=0.05,
    seed=4120,
)
import json
print(json.dumps(stats, indent=2))

{
  "lines_read": 15583,
  "records_kept": 3965,
  "split_counts": {
    "train": 3533,
    "dev": 216,
    "test": 216
  },
  "action_counts": {
    "attack": 2759,
    "spell": 1003,
    "save": 203
  },
  "unique_combats": 150
}


## 5. Inspect a few examples

In [6]:
import json, random
from fireball_preprocess import linearize_for_t5, linearize_for_ngram

with open('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/processed/train.jsonl') as f:
    lines = f.readlines()

for rec_str in random.sample(lines, k=min(3, len(lines))):
    r = json.loads(rec_str)
    print('=' * 80)
    print('ACTION:', r['action_type'], '| ACTOR:', r['actor']['name'])
    print('GOLD NARRATION:', r['narration'])
    print('\nT5 LINEARIZATION:\n  ', linearize_for_t5(r))
    print('\nN-GRAM LINEARIZATION:\n  ', linearize_for_ngram(r))
    print()

ACTION: attack | ACTOR: Ranter
GOLD NARRATION: As all of the furniture that rose up to battle our heroes now lied on the ground shattered and burned, they heard someone enter through the front door.

T5 LINEARIZATION:
   narrate | action: attack | actor: Ranter (Fighter 4, Variant Human, 40/40 hp) | target: PC2 (26→8 hp) | roll: None hit | damage: 22 unspecified, 12 unspecified | weapon: crosbow | recent: Player 0: It tried to hit marder again but failed. / Player 0: There's alot of junk but you can go down, maybe even jump over a chair or a table. You can see all the enemies. / Player 5: so question what are the names of the two plastic things in front of the flamethrower?

N-GRAM LINEARIZATION:
   <ACT>attack <ACTOR>Ranter <TGT>PC2 <HIT> <DMG>22_unspecified <DMG>12_unspecified <WPN>crosbow <NARR>

ACTION: attack | ACTOR: Biran
GOLD NARRATION: "I just had to see. This works on undead and I still don't know your name" "what that hurt, My name is Shuno Himura"

T5 LINEARIZATION:
   narr

## 6. Distribution sanity checks (for your Results section!)

In [7]:
import json
from collections import Counter

def summarize(path):
    action_c = Counter()
    narr_lens = []
    has_roll = has_dmg = has_target = 0
    total = 0
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            total += 1
            action_c[r['action_type']] += 1
            narr_lens.append(len(r['narration'].split()))
            if r['mechanics'].get('roll'): has_roll += 1
            if r['mechanics'].get('damage'): has_dmg += 1
            if r['targets']: has_target += 1
    print(f'== {path} ==')
    print(f'  total records : {total}')
    print(f'  action dist   : {dict(action_c)}')
    print(f'  narr words    : median={sorted(narr_lens)[len(narr_lens)//2] if narr_lens else 0}  mean={sum(narr_lens)/max(1,len(narr_lens)):.1f}')
    print(f'  % with roll   : {100*has_roll/max(1,total):.1f}')
    print(f'  % with damage : {100*has_dmg/max(1,total):.1f}')
    print(f'  % with target : {100*has_target/max(1,total):.1f}')
    print()

for split in ['train','dev','test']:
    summarize(f'/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/processed/{split}.jsonl')

== /Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/processed/train.jsonl ==
  total records : 3533
  action dist   : {'attack': 2458, 'spell': 888, 'save': 187}
  narr words    : median=16  mean=21.4
  % with roll   : 53.2
  % with damage : 57.6
  % with target : 84.3

== /Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/processed/dev.jsonl ==
  total records : 216
  action dist   : {'spell': 65, 'attack': 143, 'save': 8}
  narr words    : median=15  mean=19.6
  % with roll   : 53.2
  % with damage : 50.9
  % with target : 84.3

== /Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/processed/test.jsonl ==
  total records : 216
  action dist   : {'spell': 50, 'attack': 158, 'save': 8}
  narr words    : median=17  mean=22.0
  % with roll   : 56.9
  % with damage : 61.6
  % with target : 89.8



## 7. (Optional) Save to Google Drive

In [8]:
# Uncomment to mount Drive and copy over
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/cs4120_dnd
# !cp /content/processed/*.jsonl /content/drive/MyDrive/cs4120_dnd/
# !cp /content/processed/stats.json /content/drive/MyDrive/cs4120_dnd/

## Done!

Next notebooks:
- `02_train_ngram_lm.ipynb` — Week 2 baseline (N-gram with Kneser-Ney)
- `03_train_t5_seq2seq.ipynb` — Week 8 baseline (fine-tune `t5-small`)
- `04_evaluate.ipynb` — BLEU / ROUGE / BERTScore / entity-faithfulness across all models